# 01 — Data Loading & Cleaning
**Author:** Pasindu Malinda

**Pipeline:** HDFS raw CSV → Schema validation → Per-dataset cleaning → Logical integrity checks → Core join → HDFS Parquet + local CSV sample


## 1. Environment Setup & Spark Session

In [1]:
import os
import sys

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, TimestampType, BooleanType
)

spark = (
    SparkSession.builder
    .appName("ecommerce-01-data-loading-cleaning")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000")
    # keep shuffle partitions small — dataset is ~100k rows, not billions
    .config("spark.sql.shuffle.partitions", "8")
    # enable adaptive query execution (Spark 3+)
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready  |  cores: {spark.sparkContext.defaultParallelism}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 09:10:47 WARN Utils: Your hostname, Pasindus-MacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.8.188 instead (on interface en0)
26/04/22 09:10:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 09:10:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.1 ready  |  cores: 8


## 2. Path Configuration

In [2]:
HDFS_RAW     = "hdfs://localhost:9000/ecommerce/raw"
HDFS_OUTPUT  = "hdfs://localhost:9000/ecommerce/output"
LOCAL_OUTPUT = "/Users/pasindumalinda/Project-04/ecommerce-recommendation-spark/data/processed"

os.makedirs(LOCAL_OUTPUT, exist_ok=True)

print(f"HDFS raw    : {HDFS_RAW}")
print(f"HDFS output : {HDFS_OUTPUT}")
print(f"Local output: {LOCAL_OUTPUT}")

HDFS raw    : hdfs://localhost:9000/ecommerce/raw
HDFS output : hdfs://localhost:9000/ecommerce/output
Local output: /Users/pasindumalinda/Project-04/ecommerce-recommendation-spark/data/processed


## 3. Explicit Schema Definitions

Defining schemas explicitly avoids Spark's full-scan inference pass,
enforces correct types from the first read, and makes schema drift visible immediately.

In [3]:
schema_orders = StructType([
    StructField("order_id",                     StringType(),    False),
    StructField("customer_id",                  StringType(),    False),
    StructField("order_status",                 StringType(),    True),
    StructField("order_purchase_timestamp",      TimestampType(), True),
    StructField("order_approved_at",             TimestampType(), True),
    StructField("order_delivered_carrier_date",  TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True),
])

schema_order_items = StructType([
    StructField("order_id",           StringType(),    False),
    StructField("order_item_id",       IntegerType(),   True),
    StructField("product_id",          StringType(),    False),
    StructField("seller_id",           StringType(),    True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price",               DoubleType(),    True),
    StructField("freight_value",       DoubleType(),    True),
])

schema_products = StructType([
    StructField("product_id",                  StringType(),  False),
    StructField("product_category_name",        StringType(),  True),
    StructField("product_name_lenght",          DoubleType(),  True),  # typo is in source data
    StructField("product_description_lenght",   DoubleType(),  True),  # typo is in source data
    StructField("product_photos_qty",           DoubleType(),  True),
    StructField("product_weight_g",             DoubleType(),  True),
    StructField("product_length_cm",            DoubleType(),  True),
    StructField("product_height_cm",            DoubleType(),  True),
    StructField("product_width_cm",             DoubleType(),  True),
])

schema_customers = StructType([
    StructField("customer_id",             StringType(),  False),
    StructField("customer_unique_id",       StringType(),  True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city",           StringType(),  True),
    StructField("customer_state",          StringType(),  True),
])

schema_translation = StructType([
    StructField("product_category_name",         StringType(), True),
    StructField("product_category_name_english",  StringType(), True),
])

print("Schemas defined for 5 datasets")

Schemas defined for 5 datasets


## 4. Load Raw CSV Files from HDFS

In [4]:
READ_OPTS = {"header": "true", "timestampFormat": "yyyy-MM-dd HH:mm:ss"}

df_orders_raw = (
    spark.read.options(**READ_OPTS)
    .schema(schema_orders)
    .csv(f"{HDFS_RAW}/olist_orders_dataset.csv")
)

df_items_raw = (
    spark.read.options(**READ_OPTS)
    .schema(schema_order_items)
    .csv(f"{HDFS_RAW}/olist_order_items_dataset.csv")
)

df_products_raw = (
    spark.read.options(**READ_OPTS)
    .schema(schema_products)
    .csv(f"{HDFS_RAW}/olist_products_dataset.csv")
)

df_customers_raw = (
    spark.read.options(**READ_OPTS)
    .schema(schema_customers)
    .csv(f"{HDFS_RAW}/olist_customers_dataset.csv")
)

df_translation_raw = (
    spark.read.options(**READ_OPTS)
    .schema(schema_translation)
    .csv(f"{HDFS_RAW}/product_category_name_translation.csv")
)

print("All raw DataFrames loaded from HDFS")

All raw DataFrames loaded from HDFS
